# Download Mod File JSON 

In [ ]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# Lists to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None,  # Default to None (indicating missing content)
        "error_code": None  # New field for failed downloads
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        file_entry["error_code"] = "Invalid URL"
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.HTTPError as http_err:
        file_entry["error_code"] = response.status_code  # Store specific HTTP error code
        print(f"❌ Failed to fetch {direct_url} - HTTP {response.status_code}: {http_err}")
        failed_downloads.append(file_entry)
    except requests.exceptions.RequestException as e:
        file_entry["error_code"] = "Request Error"
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 0
n_rows = df.shape[0] 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"],  
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")

# Download Mod-Files (Only Annotated Ion Channels)

In [20]:
import os
import pandas as pd
import requests
from urllib.parse import urlparse, parse_qs
from tqdm import tqdm  # Progress bar

In [4]:
json_df = pd.read_json("../data/raw/mod_files.json")

In [14]:
ant_channel = pd.read_csv("../data/pipeline/preprocessed.csv", usecols=["file_hash","new_subtype_label"]).query("new_subtype_label not in ['Z Neither','Receptor']")

In [6]:
def View(df, rows=None, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`

In [17]:
json_df2 = json_df.merge(ant_channel, how="inner", on="file_hash")

In [24]:
json_df2.shape

(554, 9)

In [21]:
json_df2["new_subtype_label"].value_counts()

new_subtype_label
I K (A-type)               71
I Na (Transient)           56
I K (Delayed Rectifier)    54
I K (Ca-activated)         53
I H                        38
I Ca (L-Type HT)           38
R Glutamate (NMDA)         29
I K (M-type)               28
R GABA                     28
R Glutamate (AMPA)         26
I K (Rare)                 25
I Other (Nonspecific)      24
I Ca (T-type LT)           20
I Na (Persistent)          19
I Ca (HVA)                 12
I Other (Transporter)      12
I Na (General)             11
I Other (Leak)             10
Name: count, dtype: int64

In [28]:
json_df2

,row_id,file_hash,raw_sha,count,url,download_url,content,error_code,new_subtype_label
0,1,f8be35d0c20d1b1f3de4c44323e1780ee24f06893b6364...,67b75245b9690ce219f34a5905425672dcb5c0661a885f...,6,https://modeldb.science/183300?tab=2&file=Shor...,https://modeldb.science/getModelFile?model=183...,TITLE Sodium persistent current\n\nCOMMENT\n ...,None,I Na (Persistent)
1,2,e97ca8a7f9734805832e5ae75442d19d3d7796b9a24190...,b45c13164f1eeaf4ba61ee8522304f7382fa0541831b1a...,1,https://modeldb.science/223649?tab=2&file=Altu...,https://modeldb.science/getModelFile?model=223...,TITLE K-A channel from Klee Ficker and Heinema...,None,I K (A-type)
2,4,606423f8f4a4f406f3387c7ee6f142bd121237272c347d...,81645a02d4dbf324cf8b82e8f31bda2ce46c3753f7a8ac...,1,https://modeldb.science/262115?tab=2&file=demo...,https://modeldb.science/getModelFile?model=262...,TITLE multiple GABAa receptors\n\nCOMMENT\n---...,None,R GABA
3,5,99713c0032634e96cc7cde2dce02d8aa2baf75ad4cc835...,5fe32f050754f534d60c7c126ecd805216089cd0594062...,1,https://modeldb.science/150288?tab=2&file=KimE...,https://modeldb.science/getModelFile?model=150...,:Interneuron Cells to Pyramidal Cells GABA wit...,None,R GABA
4,11,720e1c556b6f8f457c8910a66b7594378e3aad2e835b9c...,deba2eac5d4a238c4076a5155fbc5a568f321176ffac6e...,1,https://modeldb.science/144541?tab=2&file=Ih_c...,https://modeldb.science/getModelFile?model=144...,TITLE nax\r\n: Na current for axon. No slow in...,None,I Na (Transient)
...,...,...,...,...,...,...,...,...,...
549,994,7cfd90349e150a6b279f45629a521277831975e028005c...,7b87ec2bf7624735fd1ca58f320b79a9016fed55297e62...,1,https://modeldb.science/256388?tab=2&file=Pous...,https://modeldb.science/getModelFile?model=256...,TITLE CaGk\r\n: Calcium activated K channel.\r...,None,I K (Ca-activated)
550,995,6bc782752c1bc244e268bfd48afc4bbe48017ac7298639...,8040b8be1f555ce455de9432ef8e8904eb01d664d6d17b...,6,https://modeldb.science/266802?tab=2&file=nmda...,https://modeldb.science/getModelFile?model=266...,COMMENT\r\nBiexponential single synaptic acti...,None,R Glutamate (NMDA)
551,996,d35e14f896fe05f36b2641820cca061755502705cc85f1...,c5b01637a5b008ccb68bfbe907e1e32450112dd16c9001...,1,https://modeldb.science/232023?tab=2&file=cere...,https://modeldb.science/getModelFile?model=232...,TITLE Cerebellum Granule Cell Model\n\nCOMMENT...,None,I Na (Transient)
552,997,7b79cbe8e5b42edf8939f9b1d9e4b9eef26526d8c45c41...,63061645dd5c2f1ad07b276f7d24e7ad6b01266f94bf3d...,1,https://modeldb.science/243448?tab=2&file=Mand...,https://modeldb.science/getModelFile?model=243...,TITLE TTX Sensitive Current for bladder small ...,None,I Na (Transient)


In [29]:
import os
import pandas as pd
import requests
from tqdm import tqdm

# Base output directory
base_dir = "../data/raw/nmodl"
os.makedirs(base_dir, exist_ok=True)

# Loop through the DataFrame rows
for _, row in tqdm(json_df2.iterrows(), total=len(json_df2), desc="Downloading MOD files"):
    try:
        url = row["download_url"]
        file_hash = row["file_hash"]

        # Download and save as filehash.mod
        response = requests.get(url)
        response.raise_for_status()

        output_path = os.path.join(base_dir, f"{file_hash}.mod")
        with open(output_path, "wb") as f:
            f.write(response.content)

    except Exception as e:
        print(f"Error downloading {url}: {e}")